# Orion User Activity DB Monitor

This notebook monitors Orion user activity directly from the database using a read-only Postgres connection when available.

## What this notebook covers
- User growth and signup trends
- Daily/weekly active users
- Chat session and request activity
- Model usage, token usage, and estimated cost trends
- Search usage trends
- Tier breakdowns
- Top users by activity

## Expected environment variables
This notebook prefers a direct Postgres connection via one of:
- `POSTGRES_URL_READONLY`
- `POSTGRES_URL`
- `DATABASE_URL`

If none are available, the connection cell will raise a clear error.

In [22]:
# Imports and display settings
import os
from textwrap import dedent

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Set plotly as default renderer if needed
pio.templates.default = "plotly_white"

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

In [23]:
# Database connection helpers
from pathlib import Path
from sqlalchemy import create_engine, text

# Load environment variables from a local .env if present
try:
    from dotenv import load_dotenv
    env_path = Path('.env')
    if env_path.exists():
        load_dotenv(env_path, override=False)
except ImportError:
    # If python-dotenv is unavailable, rely on already-exported env vars
    pass

DB_URL = (
    os.getenv('POSTGRES_URL_READONLY')
    or os.getenv('POSTGRES_URL')
    or os.getenv('DATABASE_URL')
)

if not DB_URL:
    raise EnvironmentError(
        'No database URL found. Set POSTGRES_URL_READONLY, POSTGRES_URL, or DATABASE_URL.'
    )

# SQLAlchemy expects the postgresql:// scheme
if DB_URL.startswith('postgres://'):
    DB_URL = DB_URL.replace('postgres://', 'postgresql://', 1)

engine = create_engine(DB_URL)


def run_query(sql: str, params: dict | None = None) -> pd.DataFrame:
    """Execute SQL and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or {})

print('Connected successfully.')

Connected successfully.


In [24]:
# Analysis window configuration
LOOKBACK_DAYS = 90

analysis_start_sql = "(current_date - (:lookback_days || ' days')::interval)"
print(f'Configured lookback window: last {LOOKBACK_DAYS} days')

Configured lookback window: last 90 days


In [25]:
# Load base tables into pandas DataFrames using simple queries
query_params = {'lookback_days': LOOKBACK_DAYS}

df_profiles = run_query("SELECT * FROM profiles")
df_models = run_query("SELECT * FROM models")

df_chat_session = run_query(
    "SELECT * FROM chat_session WHERE created_at >= current_date - (:lookback_days || ' days')::interval", 
    query_params
)
df_model_request = run_query(
    "SELECT * FROM model_request WHERE created_at >= current_date - (:lookback_days || ' days')::interval", 
    query_params
)
df_model_usage = run_query(
    "SELECT * FROM model_usage WHERE created_at >= current_date - (:lookback_days || ' days')::interval", 
    query_params
)
df_search_usage = run_query(
    "SELECT * FROM search_usage WHERE window_start >= current_date - (:lookback_days || ' days')::interval", 
    query_params
)
df_bug_reports = run_query(
    "SELECT * FROM bug_reports WHERE created_at >= current_date - (:lookback_days || ' days')::interval", 
    query_params
)

# Standardize time columns to datetime
for df in [df_profiles, df_chat_session, df_model_request, df_model_usage, df_bug_reports]:
    if 'created_at' in df.columns:
        df['created_at'] = pd.to_datetime(df['created_at'])
        
df_search_usage['window_start'] = pd.to_datetime(df_search_usage['window_start'])

# Create a unified activity dataframe for easy aggregation
activity_df = pd.concat([
    df_chat_session[['profile_id', 'created_at']].assign(source='chat_session'),
    df_model_request[['profile_id', 'created_at']].assign(source='model_request'),
    df_model_usage[['profile_id', 'created_at']].assign(source='model_usage'),
    df_search_usage[['profile_id', 'window_start']].rename(columns={'window_start': 'created_at'}).assign(source='search_usage'),
    df_bug_reports[['profile_id', 'created_at']].assign(source='bug_reports')
], ignore_index=True)

print("Loaded base tables and created unified activity DataFrame.")

Loaded base tables and created unified activity DataFrame.


## 1) Schema sanity check

In [26]:
schema_overview = run_query(dedent("""
    select table_name, column_name, data_type
    from information_schema.columns
    where table_schema = 'public'
      and table_name in (
          'profiles',
          'chat_session',
          'model_request',
          'model_usage',
          'models',
          'provider',
          'search_usage',
          'bug_reports',
          'waitlist'
      )
    order by table_name, ordinal_position
"""))

schema_overview

,table_name,column_name,data_type
0,bug_reports,id,uuid
1,bug_reports,profile_id,uuid
2,bug_reports,description,text
3,bug_reports,logs,jsonb
4,bug_reports,user_agent,text
5,bug_reports,created_at,timestamp with time zone
6,bug_reports,status,text
7,chat_session,id,uuid
8,chat_session,profile_id,uuid
9,chat_session,local_chat_id,text


## 2) Core KPI snapshot

In [27]:
# Replace SQL with pandas aggregations
now_tz = df_profiles['created_at'].dt.tz if not df_profiles.empty and df_profiles['created_at'].dt.tz else None
if now_tz:
    cutoff = pd.Timestamp.now(tz=now_tz) - pd.Timedelta(days=LOOKBACK_DAYS)
else:
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=LOOKBACK_DAYS)
    
recent_profiles = df_profiles[df_profiles['created_at'] >= cutoff] if not df_profiles.empty else df_profiles

kpi_data = [
    {'metric': 'total_profiles', 'value': len(df_profiles)},
    {'metric': 'new_profiles_last_window', 'value': len(recent_profiles)},
    {'metric': 'active_users_last_window', 'value': activity_df['profile_id'].nunique() if not activity_df.empty else 0},
    {'metric': 'chat_sessions_last_window', 'value': len(df_chat_session)},
    {'metric': 'model_requests_last_window', 'value': len(df_model_request)},
    {'metric': 'model_usage_rows_last_window', 'value': len(df_model_usage)},
    {'metric': 'tokens_in_last_window', 'value': df_model_usage['tokens_in'].sum() if not df_model_usage.empty and 'tokens_in' in df_model_usage else 0},
    {'metric': 'tokens_out_last_window', 'value': df_model_usage['tokens_out'].sum() if not df_model_usage.empty and 'tokens_out' in df_model_usage else 0},
    {'metric': 'cost_usd_last_window', 'value': df_model_usage['cost_usd'].sum() if not df_model_usage.empty and 'cost_usd' in df_model_usage else 0},
    {'metric': 'bug_reports_last_window', 'value': len(df_bug_reports)}
]
kpis = pd.DataFrame(kpi_data)
display(kpis)

,metric,value
0,total_profiles,9.000000e+00
1,new_profiles_last_window,8.000000e+00
2,active_users_last_window,5.000000e+00
3,chat_sessions_last_window,2.770000e+02
4,model_requests_last_window,1.354000e+03
5,model_usage_rows_last_window,4.155000e+03
6,tokens_in_last_window,5.978405e+07
7,tokens_out_last_window,8.239660e+05
8,cost_usd_last_window,3.747243e+01
9,bug_reports_last_window,1.000000e+01


## 3) User growth and signup trend

In [28]:
# Replace SQL with pandas group by
if not df_profiles.empty:
    now_tz = df_profiles['created_at'].dt.tz
    cutoff = (pd.Timestamp.now(tz=now_tz) - pd.Timedelta(days=LOOKBACK_DAYS)) if now_tz else (pd.Timestamp.now() - pd.Timedelta(days=LOOKBACK_DAYS))
    df_prof_recent = df_profiles[df_profiles['created_at'] >= cutoff]
    
    signups_daily = df_prof_recent.groupby(df_prof_recent['created_at'].dt.floor('D')).size().reset_index(name='signups')
    signups_daily.rename(columns={'created_at': 'day'}, inplace=True)
else:
    signups_daily = pd.DataFrame(columns=['day', 'signups'])

if not signups_daily.empty:
    fig = px.line(signups_daily, x='day', y='signups', title='Daily Signups', markers=True)
    fig.update_layout(xaxis_title='Day', yaxis_title='Signups')
    fig.show()

signups_daily.tail(20)

,day,signups
0,2026-02-16 00:00:00+00:00,1
1,2026-03-10 00:00:00+00:00,1
2,2026-03-11 00:00:00+00:00,2
3,2026-03-12 00:00:00+00:00,2
4,2026-03-30 00:00:00+00:00,1
5,2026-04-13 00:00:00+00:00,1


## 4) Daily active users (DAU) and weekly active users (WAU)

In [29]:
# Replace SQL with pandas grouping
from plotly.subplots import make_subplots

if not activity_df.empty:
    activity_df['day'] = activity_df['created_at'].dt.floor('D')
    active_users_daily = activity_df.groupby('day')['profile_id'].nunique().reset_index(name='dau')
    
    activity_df['week'] = activity_df['created_at'].dt.to_period('W').dt.start_time
    active_users_weekly = activity_df.groupby('week')['profile_id'].nunique().reset_index(name='wau')
else:
    active_users_daily = pd.DataFrame(columns=['day', 'dau'])
    active_users_weekly = pd.DataFrame(columns=['week', 'wau'])

fig = make_subplots(rows=1, cols=2, subplot_titles=('Daily Active Users', 'Weekly Active Users'))
if not active_users_daily.empty:
    fig.add_trace(go.Scatter(x=active_users_daily['day'], y=active_users_daily['dau'], mode='lines+markers', name='DAU'), row=1, col=1)
if not active_users_weekly.empty:
    fig.add_trace(go.Scatter(x=active_users_weekly['week'], y=active_users_weekly['wau'], mode='lines+markers', name='WAU'), row=1, col=2)

fig.update_layout(xaxis_title='Day', yaxis_title='Users', xaxis2_title='Week', yaxis2_title='Users', showlegend=False)
fig.show()

active_users_daily.tail(15), active_users_weekly.tail(12)

/var/folders/w_/3jkvr3rn4t30_h6hkdydn5tc0000gn/T/ipykernel_54996/3481059111.py:8: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



(                         day  dau
 33 2026-03-28 00:00:00+00:00    1
 34 2026-03-30 00:00:00+00:00    1
 35 2026-03-31 00:00:00+00:00    2
 36 2026-04-01 00:00:00+00:00    2
 37 2026-04-02 00:00:00+00:00    1
 38 2026-04-03 00:00:00+00:00    1
 39 2026-04-04 00:00:00+00:00    1
 40 2026-04-05 00:00:00+00:00    1
 41 2026-04-06 00:00:00+00:00    2
 42 2026-04-07 00:00:00+00:00    2
 43 2026-04-08 00:00:00+00:00    2
 44 2026-04-09 00:00:00+00:00    2
 45 2026-04-10 00:00:00+00:00    2
 46 2026-04-13 00:00:00+00:00    2
 47 2026-04-14 00:00:00+00:00    2,
         week  wau
 0 2026-02-16    2
 1 2026-02-23    1
 2 2026-03-02    2
 3 2026-03-09    3
 4 2026-03-16    1
 5 2026-03-23    1
 6 2026-03-30    3
 7 2026-04-06    3
 8 2026-04-13    3)

## 5) Activity by source

In [30]:
# Replace SQL with pandas value_counts
if not activity_df.empty:
    activity_by_source = activity_df['source'].value_counts().reset_index()
    activity_by_source.columns = ['source', 'events']
else:
    activity_by_source = pd.DataFrame(columns=['source', 'events'])

if not activity_by_source.empty:
    fig = px.bar(activity_by_source, x='source', y='events', title='Activity Volume by Source')
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()

activity_by_source

,source,events
0,model_usage,4155
1,model_request,1354
2,chat_session,277
3,bug_reports,10
4,search_usage,2


## 6) Chat and request trends

In [31]:
# Replace SQL with pandas merges
if not df_chat_session.empty:
    daily_chat = df_chat_session.groupby(df_chat_session['created_at'].dt.floor('D')).size().reset_index(name='chat_sessions')
else:
    daily_chat = pd.DataFrame(columns=['created_at', 'chat_sessions'])

if not df_model_request.empty:
    daily_requests = df_model_request.groupby(df_model_request['created_at'].dt.floor('D')).size().reset_index(name='model_requests')
else:
    daily_requests = pd.DataFrame(columns=['created_at', 'model_requests'])

chat_request_trends = pd.merge(daily_chat, daily_requests, on='created_at', how='outer').fillna(0)
chat_request_trends.rename(columns={'created_at': 'day'}, inplace=True)
chat_request_trends.sort_values('day', inplace=True)

if not chat_request_trends.empty:
    fig = px.line(chat_request_trends, x='day', y=['chat_sessions', 'model_requests'], title='Chat Sessions vs Model Requests', markers=True)
    fig.update_layout(xaxis_title='Day', yaxis_title='Count')
    fig.show()

chat_request_trends.tail(20)

,day,chat_sessions,model_requests
19,2026-03-23 00:00:00+00:00,10.0,31
20,2026-03-24 00:00:00+00:00,8.0,25
21,2026-03-25 00:00:00+00:00,4.0,13
22,2026-03-26 00:00:00+00:00,13.0,35
23,2026-03-27 00:00:00+00:00,17.0,52
24,2026-03-28 00:00:00+00:00,1.0,2
25,2026-03-30 00:00:00+00:00,1.0,2
26,2026-03-31 00:00:00+00:00,13.0,37
27,2026-04-01 00:00:00+00:00,14.0,36
28,2026-04-02 00:00:00+00:00,49.0,113


## 7) Model usage: tokens, cost, and model mix

In [32]:
# Replace SQL with pandas aggregation
from plotly.subplots import make_subplots

if not df_model_usage.empty:
    df_model_usage['day'] = df_model_usage['created_at'].dt.floor('D')
    
    if 'reasoning_tokens' not in df_model_usage.columns:
        df_model_usage['reasoning_tokens'] = 0
        
    usage_daily = df_model_usage.groupby('day').agg(
        usage_rows=('id', 'count') if 'id' in df_model_usage.columns else ('profile_id', 'count'),
        tokens_in=('tokens_in', 'sum'),
        tokens_out=('tokens_out', 'sum'),
        reasoning_tokens=('reasoning_tokens', 'sum'),
        cost_usd=('cost_usd', 'sum')
    ).reset_index()
else:
    usage_daily = pd.DataFrame(columns=['day', 'usage_rows', 'tokens_in', 'tokens_out', 'reasoning_tokens', 'cost_usd'])

if not usage_daily.empty:
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Daily Token Usage', 'Daily Cost (USD)'))
    fig.add_trace(go.Scatter(x=usage_daily['day'], y=usage_daily['tokens_in'], mode='lines+markers', name='Tokens In'), row=1, col=1)
    fig.add_trace(go.Scatter(x=usage_daily['day'], y=usage_daily['tokens_out'], mode='lines+markers', name='Tokens Out'), row=1, col=1)
    fig.add_trace(go.Scatter(x=usage_daily['day'], y=usage_daily['reasoning_tokens'], mode='lines+markers', name='Reasoning Tokens'), row=1, col=1)
    fig.add_trace(go.Scatter(x=usage_daily['day'], y=usage_daily['cost_usd'], mode='lines+markers', name='Cost (USD)', line=dict(color='red')), row=1, col=2)
    
    fig.update_layout(xaxis_title='Day', yaxis_title='Tokens', xaxis2_title='Day', yaxis2_title='USD')
    fig.show()

usage_daily.tail(20)

,day,usage_rows,tokens_in,tokens_out,reasoning_tokens,cost_usd
28,2026-03-23 00:00:00+00:00,150,1826285.0,31185.0,331.0,2.199575
29,2026-03-24 00:00:00+00:00,93,4152929.0,18719.0,4372.0,2.951730
30,2026-03-25 00:00:00+00:00,46,343762.0,2522.0,0.0,0.103768
31,2026-03-26 00:00:00+00:00,150,2441622.0,26961.0,5890.0,1.209933
32,2026-03-27 00:00:00+00:00,163,2586994.0,38875.0,10615.0,1.872486
33,2026-03-28 00:00:00+00:00,3,9616.0,252.0,176.0,0.003733
34,2026-03-30 00:00:00+00:00,2,1959.0,0.0,0.0,0.000000
35,2026-03-31 00:00:00+00:00,155,2812818.0,27300.0,7921.0,1.620958
36,2026-04-01 00:00:00+00:00,118,4152218.0,29661.0,0.0,1.721762
37,2026-04-02 00:00:00+00:00,394,4695807.0,52293.0,20749.0,1.280371


In [33]:
# Replace SQL with pandas aggregations
if not df_model_usage.empty:
    df_mu_m = pd.merge(df_model_usage, df_models, on='model_id', how='left')
    df_mu_m['model_label'] = df_mu_m['label'].fillna(df_mu_m['model_id'])
    
    model_mix = df_mu_m.groupby('model_label').agg(
        usage_rows=('id', 'count') if 'id' in df_mu_m.columns else ('profile_id', 'count'),
        unique_users=('profile_id', 'nunique'),
        tokens_in=('tokens_in', 'sum'),
        tokens_out=('tokens_out', 'sum'),
        cost_usd=('cost_usd', 'sum')
    ).reset_index()
    
    model_mix.sort_values(by=['cost_usd', 'usage_rows'], ascending=[False, False], inplace=True)
else:
    model_mix = pd.DataFrame(columns=['model_label', 'usage_rows', 'unique_users', 'tokens_in', 'tokens_out', 'cost_usd'])

model_mix.head(20)

,model_label,usage_rows,unique_users,tokens_in,tokens_out,cost_usd
6,GPT-5.4,821,4,23532730.0,318440.0,19.075279
3,Claude Sonnet 4.6,282,2,2631040.0,77694.0,7.532911
11,Gemini 3 Flash,772,1,11910088.0,190233.0,3.807478
12,Gemini 3.1 Flash-Lite,686,1,13673999.0,53398.0,2.217287
1,Claude Opus 4.6,38,3,356647.0,7467.0,2.129232
13,Gemini 3.1 Pro,81,1,1627011.0,47283.0,2.073215
0,Claude Haiku 4.5,46,1,291631.0,11515.0,0.262224
9,Gemini 2.5 Flash,209,1,1093780.0,16527.0,0.204535
5,GPT-5.2,515,2,3907567.0,80920.0,0.136969
20,Grok 4.20 Non-Reasoning,32,1,262863.0,1186.0,0.015708


## 8) Search usage

In [34]:
# Replace SQL with pandas merge
if not df_search_usage.empty and not df_profiles.empty:
    search_usage_snapshot = pd.merge(df_search_usage, df_profiles[['id', 'email', 'tier']], left_on='profile_id', right_on='id', how='left')
    search_usage_snapshot = search_usage_snapshot[['email', 'tier', 'profile_id', 'request_count', 'window_start']]
    search_usage_snapshot.sort_values('window_start', ascending=False, inplace=True)
else:
    search_usage_snapshot = pd.DataFrame(columns=['email', 'tier', 'profile_id', 'request_count', 'window_start'])

search_usage_snapshot.head(50)

,email,tier,profile_id,request_count,window_start
1,efonteyne@deloscapital.com.br,ultra,93489b5f-9e9e-4c25-b27e-a365ba4460a2,1,2026-04-06 14:35:34.856000+00:00
0,korissamantha@gmail.com,go,81532d8a-7595-41ba-9dc5-f1de0c3398e8,1,2026-03-10 14:52:03.881000+00:00


## 9) Tier breakdown and activity by tier

In [35]:
# Replace SQL with pandas aggregation
from plotly.subplots import make_subplots

if not df_profiles.empty:
    tier_breakdown = df_profiles['tier'].fillna('unknown').value_counts().reset_index()
    tier_breakdown.columns = ['tier', 'users']
else:
    tier_breakdown = pd.DataFrame(columns=['tier', 'users'])

if not activity_df.empty and not df_profiles.empty:
    act_prof = pd.merge(activity_df, df_profiles[['id', 'tier']], left_on='profile_id', right_on='id', how='left')
    act_prof['tier'] = act_prof['tier'].fillna('unknown')
    
    activity_by_tier = act_prof.groupby('tier').agg(
        events=('profile_id', 'count'),
        active_users=('profile_id', 'nunique')
    ).reset_index()
    activity_by_tier.sort_values('events', ascending=False, inplace=True)
else:
    activity_by_tier = pd.DataFrame(columns=['tier', 'events', 'active_users'])

fig = make_subplots(rows=1, cols=2, subplot_titles=('Users by Tier', 'Activity Events by Tier'))
if not tier_breakdown.empty:
    fig.add_trace(go.Bar(x=tier_breakdown['tier'], y=tier_breakdown['users'], name='Users'), row=1, col=1)
if not activity_by_tier.empty:
    fig.add_trace(go.Bar(x=activity_by_tier['tier'], y=activity_by_tier['events'], name='Events'), row=1, col=2)

fig.update_layout(xaxis_title='Tier', yaxis_title='Users', xaxis2_title='Tier', yaxis2_title='Events', showlegend=False)
fig.show()

tier_breakdown, activity_by_tier

(    tier  users
 0    pro      4
 1   free      2
 2  ultra      1
 3  admin      1
 4     go      1,
     tier  events  active_users
 0  admin    5272             1
 3  ultra     410             1
 1     go      93             1
 2    pro      23             2)

## 10) Top users by activity and spend

In [36]:
# Replace SQL with pandas aggregations
if not df_profiles.empty:
    sc = df_chat_session.groupby('profile_id').size().reset_index(name='chat_sessions') if not df_chat_session.empty else pd.DataFrame(columns=['profile_id', 'chat_sessions'])
    rc = df_model_request.groupby('profile_id').size().reset_index(name='model_requests') if not df_model_request.empty else pd.DataFrame(columns=['profile_id', 'model_requests'])
    
    if not df_model_usage.empty:
        ut = df_model_usage.groupby('profile_id').agg(
            usage_rows=('id', 'count') if 'id' in df_model_usage.columns else ('profile_id', 'count'),
            tokens_in=('tokens_in', 'sum'),
            tokens_out=('tokens_out', 'sum'),
            cost_usd=('cost_usd', 'sum')
        ).reset_index()
    else:
        ut = pd.DataFrame(columns=['profile_id', 'usage_rows', 'tokens_in', 'tokens_out', 'cost_usd'])
    
    top_users = df_profiles[['id', 'email', 'tier', 'created_at']].rename(columns={'id': 'profile_id', 'created_at': 'profile_created_at'})
    top_users = pd.merge(top_users, sc, on='profile_id', how='left')
    top_users = pd.merge(top_users, rc, on='profile_id', how='left')
    top_users = pd.merge(top_users, ut, on='profile_id', how='left')
    
    top_users.fillna(0, inplace=True)
    top_users = top_users[(top_users['chat_sessions'] > 0) | (top_users['model_requests'] > 0) | (top_users['usage_rows'] > 0)]
    top_users.sort_values(by=['cost_usd', 'model_requests', 'chat_sessions'], ascending=[False, False, False], inplace=True)
    top_users = top_users.head(50)
else:
    top_users = pd.DataFrame(columns=['profile_id', 'email', 'tier', 'profile_created_at', 'chat_sessions', 'model_requests', 'usage_rows', 'tokens_in', 'tokens_out', 'cost_usd'])

top_users

,profile_id,email,tier,profile_created_at,chat_sessions,model_requests,usage_rows,tokens_in,tokens_out,cost_usd
6,8d6c689d-b57f-45b4-b87d-16a9419094de,nicolasfonteyne@hotmail.com,admin,2025-12-20 08:21:39.090678+00:00,262.0,1245.0,3757.0,42324540.0,583900.0,21.437909
0,93489b5f-9e9e-4c25-b27e-a365ba4460a2,efonteyne@deloscapital.com.br,ultra,2026-03-11 14:04:34.988450+00:00,4.0,74.0,331.0,16921086.0,225819.0,14.749616
7,81532d8a-7595-41ba-9dc5-f1de0c3398e8,korissamantha@gmail.com,go,2026-02-16 15:50:47.673470+00:00,10.0,30.0,52.0,419122.0,8931.0,1.090021
5,f0df7440-306d-467a-912f-175edd2e19b2,pedro.3411@outlook.com,pro,2026-03-12 22:31:49.550802+00:00,0.0,3.0,13.0,117343.0,5316.0,0.194883
8,689adfd9-4b3d-461d-a590-51f693469017,nkarst@babson.edu,pro,2026-03-30 17:24:35.989705+00:00,1.0,2.0,2.0,1959.0,0.0,0.000000


## 11) Retention-ish view: first seen vs last seen

In [37]:
# Replace SQL with pandas aggregations
if not df_profiles.empty and not activity_df.empty:
    act_agg = activity_df.groupby('profile_id').agg(
        first_activity_at=('created_at', 'min'),
        last_activity_at=('created_at', 'max'),
        total_activity_events=('created_at', 'count')
    ).reset_index()
    
    user_lifecycle = pd.merge(df_profiles[['id', 'email', 'tier', 'created_at']], act_agg, left_on='id', right_on='profile_id', how='left')
    user_lifecycle.rename(columns={'created_at': 'signup_at'}, inplace=True)
    user_lifecycle = user_lifecycle[['profile_id', 'email', 'tier', 'signup_at', 'first_activity_at', 'last_activity_at', 'total_activity_events']]
    user_lifecycle.sort_values('last_activity_at', ascending=False, inplace=True)
else:
    user_lifecycle = pd.DataFrame(columns=['profile_id', 'email', 'tier', 'signup_at', 'first_activity_at', 'last_activity_at', 'total_activity_events'])

user_lifecycle.head(50)

,profile_id,email,tier,signup_at,first_activity_at,last_activity_at,total_activity_events
6,8d6c689d-b57f-45b4-b87d-16a9419094de,nicolasfonteyne@hotmail.com,admin,2025-12-20 08:21:39.090678+00:00,2026-02-17 04:27:44.315443+00:00,2026-04-14 21:57:23.895159+00:00,5272.0
0,93489b5f-9e9e-4c25-b27e-a365ba4460a2,efonteyne@deloscapital.com.br,ultra,2026-03-11 14:04:34.988450+00:00,2026-03-31 14:24:43.844844+00:00,2026-04-14 21:42:42.238633+00:00,410.0
7,81532d8a-7595-41ba-9dc5-f1de0c3398e8,korissamantha@gmail.com,go,2026-02-16 15:50:47.673470+00:00,2026-02-16 21:22:10.809235+00:00,2026-04-13 21:52:28.558287+00:00,93.0
8,689adfd9-4b3d-461d-a590-51f693469017,nkarst@babson.edu,pro,2026-03-30 17:24:35.989705+00:00,2026-03-30 17:28:12.863368+00:00,2026-03-30 17:28:50.934941+00:00,5.0
5,f0df7440-306d-467a-912f-175edd2e19b2,pedro.3411@outlook.com,pro,2026-03-12 22:31:49.550802+00:00,2026-03-13 00:22:45.884532+00:00,2026-03-13 00:44:14.321827+00:00,18.0
1,NaN,nicolas.fonteyne@0rion.ai,free,2026-03-10 20:51:47.049838+00:00,NaT,NaT,NaN
2,NaN,nklannfonteyne1@babson.edu,free,2026-04-13 20:30:46.533407+00:00,NaT,NaT,NaN
3,NaN,gabriel.moreira98@hotmail.com,pro,2026-03-11 17:26:13.751158+00:00,NaT,NaT,NaN
4,NaN,lucarfacciolo@gmail.com,pro,2026-03-12 00:21:41.968230+00:00,NaT,NaT,NaN


## 12) Notes

- `search_usage` appears to store the current request count per user per rolling/minute window, so it is useful as an operational snapshot but may not be a full historical event log.
- `chat_session`, `model_request`, and `model_usage` are the strongest activity signals for engagement monitoring.
- If you want, this notebook can be extended with:
  - cohort retention
  - anomaly detection
  - Slack/email alert thresholds
  - segmentation by provider/model/tier
  - a dedicated admin dashboard export

## 13) Plotly bar chart: cost per day per user

In [38]:
import plotly.express as px
import pandas as pd

# Replace SQL with pandas aggregations
if not df_model_usage.empty and not df_profiles.empty:
    df_mu_p = pd.merge(df_model_usage, df_profiles[['id', 'email']], left_on='profile_id', right_on='id', how='left')
    df_mu_p['user_label'] = df_mu_p['email'].fillna(df_mu_p['profile_id'].astype(str))
    
    user_costs = df_mu_p.groupby(['profile_id', 'user_label'])['cost_usd'].sum().reset_index()
    top_users = user_costs[user_costs['cost_usd'] > 0].sort_values('cost_usd', ascending=False).head(15)
    
    top_user_ids = top_users['profile_id'].unique()
    df_top_usage = df_mu_p[df_mu_p['profile_id'].isin(top_user_ids)].copy()
    
    df_top_usage['day'] = df_top_usage['created_at'].dt.floor('D')
    cost_per_day_per_user = df_top_usage.groupby(['day', 'user_label', 'profile_id'])['cost_usd'].sum().reset_index()
    cost_per_day_per_user = cost_per_day_per_user[cost_per_day_per_user['cost_usd'] > 0].sort_values(['day', 'cost_usd'], ascending=[True, False])
else:
    cost_per_day_per_user = pd.DataFrame(columns=['day', 'user_label', 'profile_id', 'cost_usd'])

if not cost_per_day_per_user.empty:
    cost_per_day_per_user['day'] = pd.to_datetime(cost_per_day_per_user['day'])
    cost_per_day_per_user['profile_id'] = cost_per_day_per_user['profile_id'].astype(str)
    
    fig = px.bar(
        cost_per_day_per_user,
        x='day',
        y='cost_usd',
        color='user_label',
        title='Daily Cost by User (Stacked, Top 15 Users)',
        labels={
            'day': 'Day',
            'cost_usd': 'Cost (USD)',
            'user_label': 'User'
        },
        hover_data={'profile_id': True, 'user_label': False},
    )
    
    fig.update_layout(
        barmode='stack',
        xaxis_title='Day',
        yaxis_title='Total Cost (USD)',
        legend_title='User',
        height=650,
        xaxis=dict(type='date')
    )
    
    fig.show()
else:
    print("No cost data to display.")

## 14) Cost per Chat Session

In [39]:
# Calculate cost per chat session by merging daily cost and daily chat sessions
if not usage_daily.empty and not chat_request_trends.empty:
    # Ensure 'day' is the same type for merging
    cost_per_session = pd.merge(
        usage_daily[['day', 'cost_usd']], 
        chat_request_trends[['day', 'chat_sessions']], 
        on='day', 
        how='inner'
    )
    
    # Avoid division by zero
    cost_per_session = cost_per_session[cost_per_session['chat_sessions'] > 0].copy()
    cost_per_session['cost_per_chat_session'] = cost_per_session['cost_usd'] / cost_per_session['chat_sessions']
    
    # Display the result
    display(cost_per_session.tail(10))
    
    # Plotting
    fig = px.line(cost_per_session, x='day', y='cost_per_chat_session', 
                  title='Average Cost per Chat Session (USD) over Time',
                  labels={'cost_per_chat_session': 'Cost per Session (USD)', 'day': 'Day'},
                  markers=True)
    fig.show()
else:
    print("Insufficient data to calculate cost per chat session.")

,day,cost_usd,chat_sessions,cost_per_chat_session
29,2026-04-03 00:00:00+00:00,1.715885,29.0,0.059168
30,2026-04-04 00:00:00+00:00,1.471250,14.0,0.105089
31,2026-04-05 00:00:00+00:00,0.018221,3.0,0.006073
32,2026-04-06 00:00:00+00:00,1.757830,11.0,0.159803
33,2026-04-07 00:00:00+00:00,2.228302,19.0,0.117279
34,2026-04-08 00:00:00+00:00,3.455265,2.0,1.727632
35,2026-04-09 00:00:00+00:00,1.322049,4.0,0.330512
36,2026-04-10 00:00:00+00:00,0.028421,25.0,0.001137
37,2026-04-13 00:00:00+00:00,0.293753,12.0,0.024479
38,2026-04-14 00:00:00+00:00,5.616603,4.0,1.404151


## 15) Cost of Each Actual Chat Session

In [40]:
# Calculate cost for each individual chat session
if not df_model_usage.empty and not df_model_request.empty:
    # 1. Merge Model Usage with Model Request to get 'chat_session_id'
    usage_with_session = pd.merge(
        df_model_usage[['request_id', 'cost_usd', 'tokens_in', 'tokens_out', 'model_id', 'created_at']],
        df_model_request[['id', 'chat_session_id']],
        left_on='request_id',
        right_on='id',
        how='inner'
    )
    
    # 2. Group by chat_session_id to sum the costs
    session_costs = usage_with_session.groupby('chat_session_id').agg(
        total_cost_usd=('cost_usd', 'sum'),
        total_tokens_in=('tokens_in', 'sum'),
        total_tokens_out=('tokens_out', 'sum'),
        request_count=('request_id', 'count'),
        first_request_at=('created_at', 'min'),
        last_request_at=('created_at', 'max')
    ).reset_index()
    
    # 3. Join with df_chat_session to get session metadata including local_chat_id
    detailed_session_costs = pd.merge(
        session_costs,
        df_chat_session[['id', 'profile_id', 'created_at', 'status', 'local_chat_id']],
        left_on='chat_session_id',
        right_on='id',
        how='left'
    )
    
    # 4. Order by chat creation datetime
    detailed_session_costs.sort_values(by='created_at', ascending=False, inplace=True)
    
    # Display the most recent chat sessions with their costs
    print("Most recent chat sessions with costs:")
    display(detailed_session_costs[['created_at', 'local_chat_id', 'total_cost_usd', 'request_count', 'total_tokens_in', 'total_tokens_out', 'status']].head(20))
    
    # Optional: Quick distribution plot
    if not detailed_session_costs.empty:
        fig = px.histogram(detailed_session_costs, x='total_cost_usd', 
                           nbins=50, 
                           title='Distribution of Chat Session Costs',
                           labels={'total_cost_usd': 'Total Session Cost (USD)'})
        fig.show()
else:
    print("Insufficient data to link model usage to chat sessions.")

Most recent chat sessions with costs:


,created_at,local_chat_id,total_cost_usd,request_count,total_tokens_in,total_tokens_out,status
120,2026-04-14 21:56:13.041607+00:00,1776203149126,0.034914,24,280854.0,2964.0,completed
52,2026-04-14 21:05:08.962114+00:00,1776198350660,0.004190,3,16323.0,73.0,completed
8,2026-04-14 18:13:29.534154+00:00,1776133295220,0.136207,23,348410.0,13687.0,completed
204,2026-04-14 01:55:09.697721+00:00,1776131693124,0.102333,18,310919.0,2944.0,completed
93,2026-04-13 23:31:30.973683+00:00,1776123047424,0.149604,40,752088.0,8804.0,completed
108,2026-04-13 23:26:11.462473+00:00,1776122755743,0.049155,28,196518.0,4007.0,processing
121,2026-04-13 23:19:48.683510+00:00,1776122360790,0.036346,19,135267.0,4543.0,completed
82,2026-04-13 23:16:43.402673+00:00,1776122185199,0.020926,10,68035.0,2067.0,completed
125,2026-04-13 23:13:36.771558+00:00,1776122004114,0.003051,2,11934.0,45.0,completed
25,2026-04-13 21:51:13.448912+00:00,1775599161074,0.000000,5,20497.0,504.0,processing
